# Cross-cell-type module GO/KEGG GSEA heatmap

Adapted for the selected **cross-cell-type (region-wise) modules**.

Default behavior:
- selected cross-cell-type modules: `AG_M13`, `EC_M14`, `EC_M17`, `MTC_M6`, `MTC_M13`, `PFC_M14`, `PFC_M18`;
- positive GSEA only: `NES > 0` and `FDR < 0.05`;
- only curated GO/KEGG terms in `TERM_CATEGORIES` are displayed;
- a cross-cell-type module is retained only if at least one of its cell-type components has a displayed enriched term;
- `DROP_EMPTY_CELLTYPE_COLUMNS <- TRUE` removes `region_module_celltype` columns with no displayed term;
- **each retained `region_module_celltype` column is hierarchically clustered as an independent unit by its enrichment profile across terms**;
- the **column dendrogram is drawn above the heatmap** using the exact same hierarchical clustering used for column ordering;
- region names, module names, and region grouping lines are removed from the top;
- the cell-type color strip is retained;
- x-axis labels show **cell type only**.


In [ ]:
# ============================================================
# 1. Load packages
# ============================================================

# Run once if packages are missing:
# install.packages(c("tidyverse", "ggnewscale", "scales", "grid", "svglite"))

library(tidyverse)
library(ggnewscale)
library(scales)
library(grid)


In [ ]:
# ============================================================
# 2. User settings
# ============================================================

# These are the GO/KEGG file names from your updated script.
GO_FILE <- "region_wise_GOBP_positive_p005_hits.tsv"
KEGG_FILE <- "region_wise_KEGG_positive_p005_hits.tsv"

OUTDIR <- "cross_celltype_GSEA_heatmap_ggplot"
dir.create(OUTDIR, showWarnings = FALSE, recursive = TRUE)

OUT_PREFIX <- "Figure_cross_celltype_GO_KEGG_combined_heatmap_region_module_celltype_clustered_dendrogram_regionbar"
TITLE <- "KEGG/GO enrichment across cross-cell-type regulatory modules"

REGION_ORDER <- c("AG", "EC", "HC", "MTC", "PFC", "TH")
CELLTYPE_ORDER <- c("Exc", "Inh", "Ast", "Oli", "OPC", "Mic_Immune")

# Selected cross-cell-type modules.
CROSS_CELLTYPE_MODULES <- readLines("../../part2_run_REGA/08pathology_tests/selected_module.txt", warn = FALSE)

# TRUE = exactly like the old script: after module-level filtering,
# also remove individual celltype x module columns with no displayed term.
# FALSE = keep all six cell types for every retained cross-cell-type module.
DROP_EMPTY_CELLTYPE_COLUMNS <- TRUE

region_colors <- c(
  "AG"  = "#8DA0CB",
  "EC"  = "#FC8D62",
  "HC"  = "#66C2A5",
  "MTC" = "#E78AC3",
  "PFC" = "#B3B3B3",
  "TH"  = "#A6D854"
)

celltype_colors <- c(
  "Exc"        = "#0072B2",
  "Inh"        = "#E69F00",
  "Ast"        = "#009E73",
  "Oli"        = "#D55E00",
  "OPC"        = "#CC79A7",
  "Mic_Immune" = "#6B5B53"
)

category_colors <- c(
  "Neurodegenerative disease" = "#6A51A3",
  "Immune / inflammation" = "#CB181D",
  "Mitochondrial / proteostasis / metabolism / stress" = "#2171B5",
  "Neuronal / synaptic / structural processes" = "#228833"
)


In [ ]:
# ============================================================
# 3. Figure parameters
# ============================================================

CELL_SIZE <- 0.55

LEFT_WIDTH <- 8.5
RIGHT_TEXT_UNITS <- 43
TOP_HEIGHT <- 6.5
BOTTOM_HEIGHT <- 8.0
MIN_FIG_WIDTH <- 50
MIN_FIG_HEIGHT <- 8

MAX_COLOR_VALUE <- 5.0

TITLE_FONTSIZE <- 60
TERM_TEXT_SIZE <- 55
XTICK_FONTSIZE <- 47
LEGEND_FONTSIZE <- 55
COLORBAR_FONTSIZE <- 55
CATEGORY_LEGEND_TEXT_SIZE <- 60

CATEGORY_BAR_XMAX <- 0.5
CATEGORY_BAR_WIDTH <- 0.85

# Top layout:
#   heatmap -> region color strip -> column dendrogram
REGION_BAR_Y_OFFSET <- 0.55
REGION_BAR_HEIGHT <- 0.65
DENDROGRAM_BASE_Y_OFFSET <- 0.95
DENDROGRAM_HEIGHT <- 3.50
DENDROGRAM_LINEWIDTH <- 0.75

TERM_LABEL_GAP <- 0.00
RIGHT_TERM_X <- NULL
RIGHT_LABEL_XMAX <- NULL

X_LABEL_Y <- 0.35

CATEGORY_LEGEND_TOP_GAP <- 0.90
CATEGORY_LEGEND_ROW_GAP <- 1.15
CATEGORY_LEGEND_X_SHIFT <- 1.10
CATEGORY_LEGEND_BOX_SIZE <- 0.42
CATEGORY_LEGEND_TEXT_X <- 0.65

TITLE_BOTTOM_MARGIN <- 35

MAIN_MARGIN_TOP <- 10
MAIN_MARGIN_RIGHT <- 10
MAIN_MARGIN_BOTTOM <- 10
MAIN_MARGIN_LEFT <- 20

WRITE_TABLES <- TRUE
CLUSTER_WITHIN_CATEGORY <- TRUE

# Cluster each retained region_module_celltype column as an independent unit
# using its enrichment profile across the displayed GO/KEGG terms.
CLUSTER_REGION_MODULE_CELLTYPE <- TRUE
COLUMN_CLUSTER_DISTANCE <- "euclidean"
COLUMN_CLUSTER_METHOD <- "average"

options(bitmapType = "cairo")


In [ ]:
# ============================================================
# 4. Curated KEGG/GO terms
# ============================================================

TERM_CATEGORIES <- list(
  "Neurodegenerative disease" = c(
    "ALZHEIMER DISEASE",
    "PATHWAYS OF NEURODEGENERATION - MULTIPLE DISEASES",
    "PARKINSON DISEASE",
    "HUNTINGTON DISEASE",
    "PRION DISEASE",
    "AMYOTROPHIC LATERAL SCLEROSIS"
  ),

  "Immune / inflammation" = c(
    "ANTIGEN PROCESSING AND PRESENTATION",
    "CYTOKINE-CYTOKINE RECEPTOR INTERACTION",
    "COMPLEMENT AND COAGULATION CASCADES",
    "PHAGOSOME",
    "TOLL-LIKE RECEPTOR SIGNALING PATHWAY",
    "NOD-LIKE RECEPTOR SIGNALING PATHWAY",
    "NF-KAPPA B SIGNALING PATHWAY",
    "JAK-STAT SIGNALING PATHWAY",
    "INFLAMMATORY BOWEL DISEASE",
    "STAPHYLOCOCCUS AUREUS INFECTION",
    "SYSTEMIC LUPUS ERYTHEMATOSUS",
    "AUTOIMMUNE THYROID DISEASE",
    "ALLOGRAFT REJECTION",
    "GRAFT-VERSUS-HOST DISEASE",

    "Positive Regulation of Immune System Process",
    "Regulation of Immune System Process",
    "Immune Response",
    "Inflammatory Response",
    "Positive Regulation of Cytokine Production",
    "Regulation of Cytokine Production",
    "Regulation of Chemokine Production",
    "Response to Type I Interferon",
    "Regulation of Type II Interferon Production",
    "Positive Regulation of Lymphocyte Activation",
    "Regulation of T Cell Activation",
    "Negative Regulation of T Cell Activation",
    "Regulation of T Cell Proliferation",
    "Negative Regulation of Lymphocyte Proliferation",
    "Antigen Processing and Presentation",
    "Antigen Processing and Presentation of Exogenous Peptide Antigen",
    "Antigen Processing and Presentation of Endogenous Peptide Antigen",
    "Peptide Antigen Assembly With MHC Protein Complex",
    "Macrophage Activation Involved in Immune Response",
    "Regulation of Neuroinflammatory Response",
    "Complement Activation",
    "Phagocytosis"
  ),

  "Mitochondrial / proteostasis / metabolism / stress" = c(
    "OXIDATIVE PHOSPHORYLATION",
    "MITOPHAGY - ANIMAL",
    "RIBOSOME",
    "PROTEASOME",
    "PROTEIN PROCESSING IN ENDOPLASMIC RETICULUM",
    "CHOLESTEROL METABOLISM",
    "FATTY ACID METABOLISM",
    "GLYCOLYSIS / GLUCONEOGENESIS",
    "CITRATE CYCLE (TCA CYCLE)",
    "APOPTOSIS",
    "NECROPTOSIS",
    "FERROPTOSIS",
    "CELLULAR SENESCENCE",
    "P53 SIGNALING PATHWAY",

    "Oxidative Phosphorylation",
    "Mitochondrial ATP Synthesis Coupled Electron Transport",
    "Mitochondrial Electron Transport, Cytochrome C to Oxygen",
    "Cellular Respiration",
    "ATP Synthesis Coupled Electron Transport",
    "Regulation of Release of Cytochrome C From Mitochondria",
    "Cholesterol Biosynthetic Process",
    "Lipid Biosynthetic Process",
    "Protein Folding",
    "Response to Unfolded Protein",
    "Proteasomal Protein Catabolic Process",
    "Cytoplasmic Translation",
    "Translation",
    "Ribosome Biogenesis",
    "Autophagy",
    "Mitophagy",
    "Multivesicular Body-Lysosome Fusion",
    "Regulation of Neuron Apoptotic Process",
    "Negative Regulation of Neuron Apoptotic Process"
  ),

  "Neuronal / synaptic / structural processes" = c(
    "NEUROACTIVE LIGAND-RECEPTOR INTERACTION",
    "CALCIUM SIGNALING PATHWAY",
    "GLUTAMATERGIC SYNAPSE",
    "GABAERGIC SYNAPSE",
    "CHOLINERGIC SYNAPSE",
    "DOPAMINERGIC SYNAPSE",
    "SEROTONERGIC SYNAPSE",
    "AXON GUIDANCE",
    "ECM-RECEPTOR INTERACTION",
    "CELL ADHESION MOLECULES",

    "Synapse Organization",
    "Synapse Assembly",
    "Trans-synaptic Signaling",
    "Chemical Synaptic Transmission",
    "Regulation of Synaptic Transmission, Glutamatergic",
    "Synaptic Transmission, GABAergic",
    "Axon Guidance",
    "Axonal Transport",
    "Calcium Ion Transport",
    "Extracellular Matrix Organization",
    "Cilium Organization"
  )
)


In [ ]:
TERM_CATEGORIES <- list(
  "Neurodegenerative disease" = c(
    "ALZHEIMER DISEASE",
    "PATHWAYS OF NEURODEGENERATION - MULTIPLE DISEASES",
    "PARKINSON DISEASE",
    "HUNTINGTON DISEASE",
    "PRION DISEASE",
    "AMYOTROPHIC LATERAL SCLEROSIS"
  ),

  "Immune / inflammation" = c(
    "ANTIGEN PROCESSING AND PRESENTATION",
    "CYTOKINE-CYTOKINE RECEPTOR INTERACTION",
    "COMPLEMENT AND COAGULATION CASCADES",
    "INFLAMMATORY BOWEL DISEASE",
    "AUTOIMMUNE THYROID DISEASE",
    "ALLOGRAFT REJECTION",
    "Positive Regulation of Immune System Process",
    "Regulation of T Cell Proliferation",
    "Antigen Processing and Presentation",
    #"Antigen Processing and Presentation of Exogenous Peptide Antigen",
    #"Antigen Processing and Presentation of Endogenous Peptide Antigen",
    "Peptide Antigen Assembly With MHC Protein Complex"
),

  "Mitochondrial / proteostasis / metabolism / stress" = c(
    "OXIDATIVE PHOSPHORYLATION",
    "RIBOSOME",
    "PROTEASOME",
    "Oxidative Phosphorylation",
    "Mitochondrial ATP Synthesis Coupled Electron Transport",
    "Mitochondrial Electron Transport, Cytochrome C to Oxygen",
    "Cellular Respiration",
    "ATP Synthesis Coupled Electron Transport",
    "Lipid Biosynthetic Process",
    "Protein Folding",
    "Proteasomal Protein Catabolic Process",
    "Cytoplasmic Translation",
    "Translation",
    "Ribosome Biogenesis"
  ),

  "Neuronal / synaptic / structural processes" = c(
    "NEUROACTIVE LIGAND-RECEPTOR INTERACTION",
    "GABAERGIC SYNAPSE",
    "CHOLINERGIC SYNAPSE",
    "DOPAMINERGIC SYNAPSE",
    "SEROTONERGIC SYNAPSE",
    "ECM-RECEPTOR INTERACTION",
    "Trans-synaptic Signaling",
    "Chemical Synaptic Transmission",
    #"Regulation of Synaptic Transmission, Glutamatergic",
    "Synaptic Transmission, GABAergic",
    "Axonal Transport",
    "Calcium Ion Transport",
    "Extracellular Matrix Organization",
    "Cilium Organization"
  )
)


In [ ]:
# ============================================================
# 5. Helper functions
# ============================================================

module_number <- function(module) {
  m <- stringr::str_match(as.character(module), "(\\d+)")[, 2]
  ifelse(is.na(m), 999, as.integer(m))
}

clean_go_term <- function(term) {
  term %>%
    as.character() %>%
    stringr::str_remove("\\s*\\(GO:\\d+\\)\\s*$") %>%
    stringr::str_squish()
}

normalize_term <- function(term) {
  stringr::str_to_upper(clean_go_term(term))
}

pretty_term <- function(term) {
  term <- clean_go_term(term)
  term <- ifelse(term == stringr::str_to_upper(term), stringr::str_to_title(term), term)
  term <- stringr::str_replace_all(term, c(
    "Ad" = "AD",
    "Als" = "ALS",
    "Nf-Kappa" = "NF-kappa",
    "Jak-Stat" = "JAK-STAT",
    "Tca" = "TCA",
    "Ecm" = "ECM"
  ))
  term
}

parse_celltype_region <- function(celltype_region) {
  region <- stringr::str_extract(celltype_region, "[^_]+$")
  celltype <- stringr::str_remove(celltype_region, paste0("_", region, "$"))
  celltype <- ifelse(tolower(celltype) == "mic_immune", "Mic_Immune", celltype)
  tibble(celltype = celltype, region = region)
}

parse_region_module <- function(region_module) {
  region <- stringr::str_extract(region_module, "^[^_]+")
  module <- stringr::str_extract(region_module, "M[0-9]+$")
  tibble(region = region, module = module)
}

category_label_short <- function(category) {
  dplyr::recode(
    category,
    "Neurodegenerative disease" = "Neurodegenerative disease",
    "Immune / inflammation" = "Immune / inflammation",
    "Mitochondrial / proteostasis / metabolism / stress" = "Mitochondrial / proteostasis / metabolism / stress",
    "Neuronal / synaptic / structural processes" = "Neuronal / synaptic / structural processes",
    .default = category
  )
}


### Column clustering in this version

Each retained **`region_module_celltype`** combination is treated as one independent heatmap column and hierarchically clustered by its curated GO/KEGG enrichment profile across terms. Thus, cell types from the same cross-cell-type module are **not forced to remain adjacent**.

The top of the figure now shows the **actual column dendrogram** from this clustering. Region/module text annotations and region grouping lines are removed; only the cell-type color strip is kept. Row clustering within functional categories is unchanged.


In [ ]:
# ============================================================
# 6. Build cross-cell-type column order
# ============================================================

# Each cross-cell-type module can contribute one GSEA column per cell type.
# Example for AG_M13:
#   Exc_AG_M13, Inh_AG_M13, Ast_AG_M13, Oli_AG_M13, OPC_AG_M13, Mic_Immune_AG_M13

build_column_order <- function() {
  module_info <- tibble(region_module = CROSS_CELLTYPE_MODULES) %>%
    mutate(
      region = stringr::str_extract(region_module, "^[^_]+"),
      module = stringr::str_extract(region_module, "M[0-9]+$"),
      module_order = match(region_module, CROSS_CELLTYPE_MODULES)
    )

  tidyr::crossing(
    module_info,
    celltype = CELLTYPE_ORDER
  ) %>%
    mutate(
      celltype_region = paste(celltype, region, sep = "_"),
      key = paste(celltype_region, module, sep = "_"),
      display = key,
      celltype_order = match(celltype, CELLTYPE_ORDER),
      region = factor(region, levels = REGION_ORDER),
      celltype = factor(celltype, levels = CELLTYPE_ORDER)
    ) %>%
    arrange(module_order, celltype_order) %>%
    mutate(
      col_id = row_number(),
      display = factor(display, levels = display)
    )
}

column_order_all <- build_column_order()
column_order_all


In [ ]:
# ============================================================
# 7. Read GSEA files and select curated terms
# ============================================================

read_gsea_file <- function(path) {
  df <- readr::read_tsv(path, show_col_types = FALSE)

  # Required GSEA result columns.
  required <- c("module", "term", "nes", "fdr")
  missing <- setdiff(required, colnames(df))
  if (length(missing) > 0) {
    stop(path, " is missing columns: ", paste(missing, collapse = ", "))
  }

  # The existing result files use celltype_region. This fallback also accepts
  # separate celltype + region columns if a later file is written that way.
  if (!"celltype_region" %in% colnames(df)) {
    if (all(c("celltype", "region") %in% colnames(df))) {
      df <- df %>% mutate(celltype_region = paste(celltype, region, sep = "_"))
    } else {
      stop(path, " must contain either celltype_region or both celltype and region.")
    }
  }

  parsed <- parse_celltype_region(df$celltype_region)

  df %>%
    mutate(
      module = as.character(module),
      celltype = parsed$celltype,
      region = parsed$region,
      region_module = paste(region, module, sep = "_"),
      key = paste(celltype_region, module, sep = "_"),
      term_clean = clean_go_term(term),
      term_match = normalize_term(term),
      nes = as.numeric(nes),
      fdr = as.numeric(fdr)
    ) %>%
    filter(
      region_module %in% CROSS_CELLTYPE_MODULES,
      celltype %in% CELLTYPE_ORDER,
      nes > 0,
      fdr < 0.05
    )
}

collect_selected_hits <- function(go_file, kegg_file, column_order) {
  dfs <- list()

  if (file.exists(go_file)) {
    dfs[[length(dfs) + 1]] <- read_gsea_file(go_file)
  } else {
    message("GO file not found: ", go_file)
  }

  if (file.exists(kegg_file)) {
    dfs[[length(dfs) + 1]] <- read_gsea_file(kegg_file)
  } else {
    message("KEGG file not found: ", kegg_file)
  }

  if (length(dfs) == 0) {
    stop("No GO or KEGG input files found.")
  }

  df <- bind_rows(dfs) %>%
    filter(key %in% column_order$key)

  term_map <- purrr::imap_dfr(TERM_CATEGORIES, function(terms, category) {
    tibble(
      category = category,
      term_original = terms,
      term_match = normalize_term(terms),
      plot_term = pretty_term(terms)
    )
  }) %>%
    distinct(term_match, .keep_all = TRUE)

  out <- df %>%
    left_join(term_map, by = "term_match") %>%
    filter(!is.na(category), !is.na(plot_term)) %>%
    mutate(category = factor(category, levels = names(TERM_CATEGORIES))) %>%
    group_by(category, plot_term, term_match, key, region_module) %>%
    summarise(
      best_fdr = min(fdr, na.rm = TRUE),
      max_nes = max(nes, na.rm = TRUE),
      .groups = "drop"
    ) %>%
    mutate(
      score = -log10(pmax(best_fdr, 1e-300)),
      score = pmin(score, MAX_COLOR_VALUE)
    )

  if (nrow(out) == 0) {
    stop(
      "No selected cross-cell-type modules have curated positive GO/KEGG terms ",
      "with NES > 0 and FDR < 0.05."
    )
  }

  out
}

selected_hits <- collect_selected_hits(GO_FILE, KEGG_FILE, column_order_all)

cat("Curated enriched terms by category:\n")
print(selected_hits %>% count(category))

cat("\nCross-cell-type modules with at least one displayed enriched term:\n")
print(
  selected_hits %>%
    distinct(region_module) %>%
    arrange(match(region_module, CROSS_CELLTYPE_MODULES))
)


In [ ]:
# ============================================================
# 8. Keep only cross-cell-type modules that have enriched terms
# ============================================================

# Module-level filtering: if none of the cell-type components of a region_module
# has a selected GO/KEGG term, remove the entire cross-cell-type module.
enriched_region_modules <- selected_hits %>%
  distinct(region_module) %>%
  pull(region_module)

enriched_region_modules <- CROSS_CELLTYPE_MODULES[
  CROSS_CELLTYPE_MODULES %in% enriched_region_modules
]

if (length(enriched_region_modules) == 0) {
  stop("None of the selected cross-cell-type modules has a displayed enriched term.")
}

column_order <- column_order_all %>%
  filter(region_module %in% enriched_region_modules) %>%
  arrange(match(region_module, CROSS_CELLTYPE_MODULES), match(as.character(celltype), CELLTYPE_ORDER)) %>%
  mutate(
    col_id = row_number(),
    display = factor(as.character(display), levels = as.character(display))
  )

selected_hits <- selected_hits %>%
  filter(region_module %in% enriched_region_modules)

cat("Kept cross-cell-type modules:\n")
print(enriched_region_modules)
cat("\nInitial celltype x module columns after module-level filtering:", nrow(column_order), "\n")


In [ ]:
# ============================================================
# 9. Build heatmap matrix, cluster rows, and cluster columns
# ============================================================

build_matrix_and_rows <- function(selected_hits, column_order) {
  row_info <- selected_hits %>%
    distinct(category, plot_term, term_match) %>%
    mutate(category = factor(category, levels = names(TERM_CATEGORIES))) %>%
    arrange(category, plot_term)

  mat <- selected_hits %>%
    select(term_match, key, score) %>%
    tidyr::complete(
      term_match = row_info$term_match,
      key = column_order$key,
      fill = list(score = 0)
    ) %>%
    group_by(term_match, key) %>%
    summarise(score = max(score), .groups = "drop") %>%
    pivot_wider(names_from = key, values_from = score, values_fill = 0) %>%
    arrange(match(term_match, row_info$term_match))

  mat_df <- as.data.frame(mat)
  rownames(mat_df) <- mat_df$term_match
  mat_df$term_match <- NULL
  mat_df <- mat_df[, column_order$key, drop = FALSE]

  list(mat = as.matrix(mat_df), row_info = row_info)
}

cluster_rows_within_category <- function(mat, row_info) {
  if (!CLUSTER_WITHIN_CATEGORY) {
    return(list(mat = mat, row_info = row_info))
  }

  new_order <- c()

  for (cat in names(TERM_CATEGORIES)) {
    idx <- which(as.character(row_info$category) == cat)

    if (length(idx) <= 2) {
      new_order <- c(new_order, idx)
      next
    }

    sub <- mat[idx, , drop = FALSE]

    # If all rows are identical, do not cluster this category.
    row_strings <- apply(sub, 1, paste, collapse = ":")
    if (length(unique(row_strings)) <= 1) {
      new_order <- c(new_order, idx)
      next
    }

    hc <- tryCatch(
      hclust(dist(sub), method = "average"),
      error = function(e) NULL
    )

    if (is.null(hc)) {
      new_order <- c(new_order, idx)
    } else {
      new_order <- c(new_order, idx[hc$order])
    }
  }

  list(
    mat = mat[new_order, , drop = FALSE],
    row_info = row_info[new_order, , drop = FALSE]
  )
}

drop_empty_columns <- function(mat, column_order) {
  keep_cols <- colSums(mat, na.rm = TRUE) > 0

  if (!any(keep_cols)) {
    stop("All columns have zero enrichment after filtering curated terms.")
  }

  kept_keys <- colnames(mat)[keep_cols]

  column_order2 <- column_order %>%
    filter(key %in% kept_keys) %>%
    mutate(
      col_id = row_number(),
      display = factor(as.character(display), levels = as.character(display))
    )

  mat2 <- mat[, column_order2$key, drop = FALSE]

  list(mat = mat2, column_order = column_order2)
}

cluster_region_module_celltype_columns <- function(mat, column_order) {
  # Each heatmap column is one independent region_module_celltype unit.
  # Similarity is computed from that column's vector of enrichment scores
  # across all displayed terms.
  hc_columns <- NULL

  if (!CLUSTER_REGION_MODULE_CELLTYPE || ncol(mat) <= 1) {
    ordered_keys <- colnames(mat)
  } else {
    column_profiles <- t(mat)

    # If all column profiles are identical, keep the current plotting order.
    profile_strings <- apply(column_profiles, 1, paste, collapse = ":")

    if (length(unique(profile_strings)) <= 1) {
      ordered_keys <- rownames(column_profiles)
    } else {
      hc_columns <- tryCatch(
        hclust(
          dist(column_profiles, method = COLUMN_CLUSTER_DISTANCE),
          method = COLUMN_CLUSTER_METHOD
        ),
        error = function(e) NULL
      )

      if (is.null(hc_columns)) {
        warning("region_module_celltype clustering failed; retaining the current column order.")
        ordered_keys <- rownames(column_profiles)
      } else {
        ordered_keys <- rownames(column_profiles)[hc_columns$order]
      }
    }
  }

  column_order2 <- column_order %>%
    mutate(cluster_order = match(key, ordered_keys)) %>%
    arrange(cluster_order) %>%
    mutate(
      region = factor(as.character(region), levels = REGION_ORDER),
      celltype = factor(as.character(celltype), levels = CELLTYPE_ORDER),
      col_id = row_number(),
      display = factor(as.character(display), levels = as.character(display))
    ) %>%
    select(-cluster_order)

  mat2 <- mat[, column_order2$key, drop = FALSE]

  list(
    mat = mat2,
    column_order = column_order2,
    ordered_keys = ordered_keys,
    hclust = hc_columns
  )
}

# Convert an hclust object to line segments for a ggplot dendrogram.
# Leaf x positions are exactly 1:ncol in the clustered heatmap order.
hclust_to_segments <- function(hc) {
  if (is.null(hc) || length(hc$order) <= 1) {
    return(tibble(x = numeric(), xend = numeric(), height = numeric(), heightend = numeric()))
  }

  n_leaves <- length(hc$order)
  leaf_x <- rep(NA_real_, n_leaves)
  leaf_x[hc$order] <- seq_len(n_leaves)

  node_x <- rep(NA_real_, nrow(hc$merge))
  node_h <- hc$height
  seg_list <- vector("list", nrow(hc$merge))

  child_xy <- function(child, current_i) {
    if (child < 0) {
      leaf_id <- -child
      return(c(x = leaf_x[leaf_id], h = 0))
    }

    # Positive values refer to an earlier merge row.
    c(x = node_x[child], h = node_h[child])
  }

  for (i in seq_len(nrow(hc$merge))) {
    left <- child_xy(hc$merge[i, 1], i)
    right <- child_xy(hc$merge[i, 2], i)
    parent_h <- node_h[i]
    parent_x <- mean(c(left["x"], right["x"]))
    node_x[i] <- parent_x

    seg_list[[i]] <- tibble(
      x = c(left["x"], right["x"], left["x"]),
      xend = c(left["x"], right["x"], right["x"]),
      height = c(left["h"], right["h"], parent_h),
      heightend = c(parent_h, parent_h, parent_h)
    )
  }

  bind_rows(seg_list)
}

built <- build_matrix_and_rows(selected_hits, column_order)
clustered_rows <- cluster_rows_within_category(built$mat, built$row_info)

mat <- clustered_rows$mat
row_info <- clustered_rows$row_info

# Remove empty plotting units before column clustering when requested.
if (DROP_EMPTY_CELLTYPE_COLUMNS) {
  n_before_drop <- ncol(mat)
  nonempty <- drop_empty_columns(mat, column_order)
  mat <- nonempty$mat
  column_order <- nonempty$column_order
  message(
    "region_module_celltype columns kept after removing empty columns: ",
    ncol(mat), " / ", n_before_drop
  )
} else {
  message("Keeping all region_module_celltype columns: ", ncol(mat))
}

# Cluster every retained region_module_celltype combination independently.
clustered_columns_result <- cluster_region_module_celltype_columns(mat, column_order)
mat <- clustered_columns_result$mat
column_order <- clustered_columns_result$column_order
COLUMN_HCLUST <- clustered_columns_result$hclust
CLUSTERED_REGION_MODULE_CELLTYPE_ORDER <- clustered_columns_result$ordered_keys

cat("region_module_celltype order used in heatmap:\n")
print(CLUSTERED_REGION_MODULE_CELLTYPE_ORDER)

# Module summary after all filtering. Because columns are independently clustered,
# components of the same module may occur in separate parts of the heatmap.
kept_module_report <- column_order %>%
  mutate(celltype = as.character(celltype), region = as.character(region)) %>%
  group_by(region_module, region, module) %>%
  summarise(
    n_celltype_columns = n(),
    celltypes_shown = paste(celltype, collapse = ", "),
    first_col = min(col_id),
    .groups = "drop"
  ) %>%
  arrange(first_col) %>%
  select(-first_col)

kept_module_report
row_info %>% count(category)


In [ ]:
# ============================================================
# 10. Convert matrix to ggplot data
# ============================================================

make_plot_data <- function(mat, row_info, column_order, column_hclust = NULL) {
  n_rows <- nrow(row_info)
  n_cols <- nrow(column_order)

  row_info2 <- row_info %>%
    mutate(
      row_id = row_number(),
      y = n_rows - row_id + 1,
      category = factor(category, levels = names(TERM_CATEGORIES))
    )

  heatmap_df <- as.data.frame(mat) %>%
    rownames_to_column("term_match") %>%
    pivot_longer(-term_match, names_to = "key", values_to = "score") %>%
    left_join(row_info2, by = "term_match") %>%
    left_join(column_order, by = "key") %>%
    mutate(x = col_id)

  category_ranges <- row_info2 %>%
    group_by(category) %>%
    summarise(
      ymin = min(y) - 0.5,
      ymax = max(y) + 0.5,
      ymid = (ymin + ymax) / 2,
      .groups = "drop"
    ) %>%
    mutate(
      xmin = CATEGORY_BAR_XMAX - CATEGORY_BAR_WIDTH,
      xmax = CATEGORY_BAR_XMAX
    )

  # Build dendrogram segments from the SAME hclust object used to order columns.
  dendrogram_df <- hclust_to_segments(column_hclust)

  if (nrow(dendrogram_df) > 0) {
    max_h <- max(c(dendrogram_df$height, dendrogram_df$heightend), na.rm = TRUE)
    dendro_base_y <- n_rows + DENDROGRAM_BASE_Y_OFFSET

    if (is.finite(max_h) && max_h > 0) {
      dendrogram_df <- dendrogram_df %>%
        mutate(
          y = dendro_base_y + (height / max_h) * DENDROGRAM_HEIGHT,
          yend = dendro_base_y + (heightend / max_h) * DENDROGRAM_HEIGHT
        )
    } else {
      dendrogram_df <- dendrogram_df %>%
        mutate(y = dendro_base_y, yend = dendro_base_y)
    }
  } else {
    dendrogram_df <- tibble(
      x = numeric(), xend = numeric(),
      height = numeric(), heightend = numeric(),
      y = numeric(), yend = numeric()
    )
  }

  list(
    heatmap_df = heatmap_df,
    row_info = row_info2,
    column_order = column_order,
    category_ranges = category_ranges,
    dendrogram_df = dendrogram_df,
    n_rows = n_rows,
    n_cols = n_cols
  )
}

plot_data <- make_plot_data(mat, row_info, column_order, COLUMN_HCLUST)
str(plot_data, max.level = 1)


In [ ]:
# ============================================================
# 11. Plot functions
# ============================================================

make_heatmap_plot <- function(plot_data) {
  heatmap_df <- plot_data$heatmap_df
  row_info <- plot_data$row_info
  column_order <- plot_data$column_order
  category_ranges <- plot_data$category_ranges
  dendrogram_df <- plot_data$dendrogram_df
  n_rows <- plot_data$n_rows
  n_cols <- plot_data$n_cols

  # Top annotation positions.
  region_bar_y <- n_rows + REGION_BAR_Y_OFFSET
  y_top <- n_rows + DENDROGRAM_BASE_Y_OFFSET + DENDROGRAM_HEIGHT + 0.35

  # First letter of each term starts at the heatmap right edge.
  # Heatmap tile centers are 1:n_cols, so right edge is n_cols + 0.5.
  term_x <- n_cols + 0.5 + TERM_LABEL_GAP
  x_right <- n_cols + RIGHT_TEXT_UNITS

  region_bar_df <- column_order %>%
    mutate(
      x = col_id,
      y = region_bar_y,
      region = factor(as.character(region), levels = REGION_ORDER)
    )

  # X-axis labels identify each independently clustered unit as region_M_celltype.
  # Example: AG_M13_Exc.
  x_label_df <- column_order %>%
    mutate(
      x = col_id,
      y = X_LABEL_Y,
      label = paste(as.character(region), module, as.character(celltype), sep = "_")
    )

  term_df <- row_info %>%
    transmute(
      x = term_x,
      y = y,
      plot_term = plot_term
    )

  category_legend_df <- tibble(
    category = names(TERM_CATEGORIES),
    label = category_label_short(names(TERM_CATEGORIES)),
    item = seq_along(names(TERM_CATEGORIES))
  ) %>%
    mutate(
      category = factor(category, levels = names(TERM_CATEGORIES)),
      x_box = term_x + CATEGORY_LEGEND_X_SHIFT,
      x_text = x_box + CATEGORY_LEGEND_TEXT_X,
      y = 0.5 - CATEGORY_LEGEND_TOP_GAP - (item - 1) * CATEGORY_LEGEND_ROW_GAP
    )

  y_bottom <- min(category_legend_df$y, x_label_df$y) - 0.8

  ggplot() +
    # Category color bar, attached directly to the heatmap left edge.
    # No category text on the left; category names are shown in the legend under terms.
    geom_rect(
      data = category_ranges,
      aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax, fill = category),
      color = NA
    ) +
    scale_fill_manual(values = category_colors, guide = "none") +

    ggnewscale::new_scale_fill() +

    # Main heatmap. geom_tile keeps PDF/SVG crisp; no raster imshow blur.
    geom_tile(
      data = heatmap_df,
      aes(x = x, y = y, fill = score),
      width = 1,
      height = 1,
      color = "#BDBDBD",
      linewidth = 0.45
    ) +
    scale_fill_gradientn(
      colours = c("#FFFFFF", "#FEE0D2", "#FC9272", "#DE2D26", "#A50F15"),
      limits = c(0, MAX_COLOR_VALUE),
      oob = scales::squish,
      name = expression(-log[10](FDR)),
      guide = guide_colorbar(
        order = 1,
        title.position = "right",
        barheight = unit(3.5, "in"),
        barwidth = unit(0.35, "in")
      )
    ) +

    ggnewscale::new_scale_fill() +

    # Region color strip directly above the heatmap.
    geom_tile(
      data = region_bar_df,
      aes(x = x, y = y, fill = region),
      width = 1,
      height = REGION_BAR_HEIGHT,
      color = "white",
      linewidth = 0.35
    ) +
    scale_fill_manual(
      values = region_colors,
      name = "Region",
      drop = FALSE,
      guide = guide_legend(order = 2, override.aes = list(size = 8))
    ) +

    # Column dendrogram from the exact same hclust used for column ordering.
    # Region/module names and region grouping lines are intentionally omitted.
    geom_segment(
      data = dendrogram_df,
      aes(x = x, xend = xend, y = y, yend = yend),
      inherit.aes = FALSE,
      linewidth = DENDROGRAM_LINEWIDTH,
      lineend = "square",
      color = "black"
    ) +

    # Term labels: first letter starts right next to the heatmap right edge.
    geom_text(
      data = term_df,
      aes(x = x, y = y, label = plot_term),
      hjust = 0,
      vjust = 0.5,
      size = TERM_TEXT_SIZE / .pt,
      color = "black"
    ) +

    # Manually drawn x-axis labels, placed directly below the heatmap.
    # Each label is region_module_celltype, e.g. AG_M13_Exc.
    geom_text(
      data = x_label_df,
      aes(x = x, y = y, label = label),
      angle = 90,
      hjust = 1,
      vjust = 0.5,
      size = XTICK_FONTSIZE / .pt,
      color = "black"
    ) +

    # Category legend under the last term, aligned with the term labels.
    ggnewscale::new_scale_fill() +
    geom_tile(
      data = category_legend_df,
      aes(x = x_box, y = y, fill = category),
      width = CATEGORY_LEGEND_BOX_SIZE,
      height = CATEGORY_LEGEND_BOX_SIZE,
      color = NA,
      show.legend = FALSE
    ) +
    geom_text(
      data = category_legend_df,
      aes(x = x_text, y = y, label = label),
      hjust = 0,
      vjust = 0.5,
      size = CATEGORY_LEGEND_TEXT_SIZE / .pt,
      fontface = "bold",
      lineheight = 0.85,
      color = "black",
      show.legend = FALSE
    ) +
    scale_fill_manual(values = category_colors, guide = "none") +

    scale_x_continuous(
      breaks = NULL,
      labels = NULL,
      limits = c(CATEGORY_BAR_XMAX - CATEGORY_BAR_WIDTH, x_right),
      expand = expansion(mult = c(0, 0))
    ) +
    scale_y_continuous(
      breaks = NULL,
      labels = NULL,
      limits = c(y_bottom, y_top),
      expand = expansion(mult = c(0, 0))
    ) +

    coord_fixed(ratio = 1, clip = "off") +

    labs(title = TITLE, x = NULL, y = NULL) +

    theme_minimal(base_size = 12) +
    theme(
      panel.grid = element_blank(),
      panel.background = element_blank(),
      plot.background = element_blank(),

      axis.text.x = element_blank(),
      axis.text.y = element_blank(),
      axis.ticks = element_blank(),

      plot.title = element_text(
        size = TITLE_FONTSIZE,
        face = "bold",
        hjust = 0.5,
        margin = margin(b = TITLE_BOTTOM_MARGIN)
      ),

      legend.position = "left",
      legend.box = "vertical",
      legend.title = element_text(size = LEGEND_FONTSIZE, face = "bold"),
      legend.text = element_text(size = LEGEND_FONTSIZE),
      legend.margin = margin(0, 10, 0, 0),

      plot.margin = margin(
        t = MAIN_MARGIN_TOP,
        r = MAIN_MARGIN_RIGHT,
        b = MAIN_MARGIN_BOTTOM,
        l = MAIN_MARGIN_LEFT
      )
    )
}

make_final_plot <- function(plot_data) {
  make_heatmap_plot(plot_data)
}


In [ ]:
# ============================================================
# 12. Preview plot in notebook
# ============================================================

final_plot <- make_final_plot(plot_data)
#final_plot


In [ ]:
# ============================================================
# 13. Save outputs
# ============================================================

save_outputs <- function(final_plot, plot_data, mat, row_info, column_order, selected_hits, kept_module_report) {
  n_rows <- plot_data$n_rows
  n_cols <- plot_data$n_cols

  fig_width <- max(
    MIN_FIG_WIDTH,
    LEFT_WIDTH + (n_cols + RIGHT_TEXT_UNITS) * CELL_SIZE
  )

  fig_height <- max(
    MIN_FIG_HEIGHT,
    TOP_HEIGHT + n_rows * CELL_SIZE + BOTTOM_HEIGHT + 20
  )

  pdf_file <- file.path(OUTDIR, paste0(OUT_PREFIX, ".pdf"))
  png_file <- file.path(OUTDIR, paste0(OUT_PREFIX, ".png"))
  svg_file <- file.path(OUTDIR, paste0(OUT_PREFIX, ".svg"))

  tryCatch(
    {
      ggsave(
        pdf_file,
        plot = final_plot,
        width = fig_width,
        height = fig_height,
        units = "in",
        device = grDevices::cairo_pdf,
        limitsize = FALSE
      )
    },
    error = function(e) {
      message("cairo_pdf failed; falling back to standard pdf device.")
      ggsave(
        pdf_file,
        plot = final_plot,
        width = fig_width,
        height = fig_height,
        units = "in",
        device = "pdf",
        useDingbats = FALSE,
        limitsize = FALSE
      )
    }
  )
  fig_width=fig_width+1.7
  ggsave(
    png_file,
    plot = final_plot,
    width = fig_width,
    height = fig_height,
    units = "in",
    dpi = 300,
    limitsize = FALSE
  )

  if (requireNamespace("svglite", quietly = TRUE)) {
    ggsave(
      svg_file,
      plot = final_plot,
      width = fig_width,
      height = fig_height,
      units = "in",
      device = svglite::svglite,
      limitsize = FALSE
    )
  } else {
    message("Package 'svglite' is not installed. Skipping SVG output.")
    svg_file <- NA_character_
  }

  if (WRITE_TABLES) {
    readr::write_tsv(column_order, file.path(OUTDIR, "region_module_celltype_clustered_dendrogram_column_order.tsv"))
    readr::write_tsv(kept_module_report, file.path(OUTDIR, "cross_celltype_modules_kept.tsv"))
    readr::write_tsv(selected_hits, file.path(OUTDIR, "selected_GO_KEGG_hits_for_heatmap.tsv"))
    readr::write_tsv(row_info, file.path(OUTDIR, "heatmap_row_order.tsv"))
    readr::write_tsv(
      as.data.frame(mat) %>% rownames_to_column("term_match"),
      file.path(OUTDIR, "heatmap_matrix_minus_log10_FDR.tsv")
    )
  }

  message("Saved files:")
  message(pdf_file)
  message(png_file)
  if (!is.na(svg_file)) message(svg_file)

  invisible(na.omit(c(pdf_file, png_file, svg_file)))
}

save_outputs(
  final_plot,
  plot_data,
  mat,
  row_info,
  column_order,
  selected_hits,
  kept_module_report
)
